# Stage 2 — Balanced-Sampling Fix, Full Training Grid & Evaluation

**Single entry-point notebook.** Every cell is **idempotent**: interrupt the kernel, restart, click **Run All**, and it picks up exactly where it left off (completed runs detected via `metrics.json`).

### Pipeline overview
1. **Migrations** — resolve synthetic layout, link legacy checkpoints. Synthetic images are **already generated**; this cell never triggers regeneration.
2. **Validation sanity-check** — Uniform 5× ResNet-18 (30 ep). Confirms the balanced-sampling fix works (expect ≥ 40 %).
3. **Tiny ImageNet ResNet-18 full grid** — baseline (skip if done) → uniform 5×/10×/15× → diagnostics → utility targets → allocation CSVs → adaptive (hard_class / uncertainty / predicted_utility) → ceiling. **All standard CE.**
4. **Tiny ImageNet MobileNetV3-Small** — baseline → uniform 15× → adaptive predicted_utility → ceiling. **All standard CE.**
5. **CIFAR-100 ResNet-18** — baseline → uniform 15× → diagnostics + utility + allocations → adaptive predicted_utility → ceiling. **All standard CE.**
6. **FID** — read cached `fid_summary.json`; only recompute if missing.
7. **Figures** — summary bar chart + any per-run plots, from `results/` into `figures/stage2/`.
8. **Synthetic-aware loss (optional)** — flip `RUN_SA_VARIANTS = True`, re-Run All, and only the new `_sa` pipelines train. Everything else is skipped.

**Prerequisites:** Stage 1 subset (`data/tiny_imagenet_5pct/train_index.csv`), raw datasets, synthetic pools already generated under `data/synthetic/`.

In [1]:
# ============================================================
# 0.  Setup — imports, paths, reproducibility, master switches
# ============================================================
import os, sys, json, importlib, random
from pathlib import Path

import numpy as np
import torch

# ---- project root (notebook lives in notebooks/) ----
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

# ---- reproducibility ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ---- master switches (all True = full end-to-end) ----
RUN_MIGRATION          = True   # symlinks; never regenerates synthetic images
RUN_VALIDATION_CHECK   = True   # quick Uniform 5× R18 sanity test
RUN_TINY_R18           = True   # full Tiny ImageNet ResNet-18 grid (standard CE)
RUN_TINY_MOBILENET     = True   # Tiny ImageNet MobileNetV3-Small subset (standard CE)
RUN_CIFAR              = True   # CIFAR-100 ResNet-18 track (standard CE)
RUN_FID                = True   # global FID (reads cache first; only computes if missing)
RUN_FIGURES            = True   # summary plots from results/
RUN_SA_VARIANTS        = False  # ← flip to True later to add synthetic-aware loss runs

print(f"Project root : {PROJECT_ROOT}")
print(f"CUDA         : {torch.cuda.is_available()}", end="")
if torch.cuda.is_available():
    print(f"  ({torch.cuda.get_device_name(0)})")
else:
    print("  (cpu)")

Project root : /mnt/data/cv
CUDA         : True  (NVIDIA GeForce RTX 4060 Ti)


## I. Orchestrator & helpers

Force-reload every `src.*` module that may have been edited between kernel restarts, then build the shared helpers used by every section below.

In [2]:
# ---- force-reload all src modules so on-disk fixes always take effect ----
_reload_order = [
    "src.models.backbone",
    "src.evaluation.eval_extras",
    "src.stage2.diagnostics",
    "src.training.synthetic_loss",
    "src.training.train_eval",
    "src.training.stage2_train",
    "src.evaluation.stage2_eval",
    "src.evaluation.fid_stage2",
    "src.data.synthetic_dataset",
    "src.data.registry",
    "src.allocation.policies",
    "src.stage2.orchestrator",
]
for _mn in _reload_order:
    if _mn in sys.modules:
        importlib.reload(sys.modules[_mn])

from src.stage2.orchestrator import Stage2Orchestrator
from src.config import load_experiment_config
from src.models.backbone import build_backbone
from src.evaluation.stage2_eval import evaluate_stage2
from src.data.registry import class_ids_in_label_order, get_baseline_loaders, build_real_train_subset
from src.data.transforms import get_train_transform, get_val_transform

orch = Stage2Orchestrator(PROJECT_ROOT)

# ---- idempotent run finder (shared by every section) ----
def find_completed_run(dataset: str, pipeline: str, arch: str) -> tuple:
    """Return (run_dir, metrics_dict) for the latest run that has metrics.json.
    Returns (None, {}) if nothing completed yet."""
    parent = PROJECT_ROOT / "results" / dataset / pipeline / arch
    if not parent.is_dir():
        return None, {}
    candidates = sorted(
        [d for d in parent.iterdir() if d.is_dir() and (d / "metrics.json").is_file()],
        key=lambda d: d.name,
    )
    if not candidates:
        # Fallback: best.pt exists but metrics.json missing → need re-eval
        partial = sorted(
            [d for d in parent.iterdir() if d.is_dir() and (d / "best.pt").is_file()],
            key=lambda d: d.name,
        )
        if partial:
            return partial[-1], {}  # caller should re-evaluate
        return None, {}
    rd = candidates[-1]
    m = json.loads((rd / "metrics.json").read_text(encoding="utf-8"))
    return rd, m


def ensure_eval(rd: Path, arch: str, cfg_yaml: str, baseline_ckpt: Path = None) -> dict:
    """Re-evaluate a run that trained (best.pt) but has no metrics.json."""
    cfg = load_experiment_config(orch.config_path(cfg_yaml), PROJECT_ROOT)
    tr_t = get_train_transform(cfg.dataset.image_size)
    va_t = get_val_transform(cfg.dataset.image_size)
    _, val_loader, c2i = get_baseline_loaders(cfg, tr_t, va_t)
    model = build_backbone(arch, cfg.dataset.num_classes)
    model.load_state_dict(torch.load(rd / "best.pt", map_location="cpu", weights_only=True))
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(dev)
    ref_cka = None
    if baseline_ckpt and baseline_ckpt.is_file():
        ref_cka = build_backbone(arch, cfg.dataset.num_classes)
        ref_cka.load_state_dict(torch.load(baseline_ckpt, map_location="cpu", weights_only=True))
        ref_cka.to(dev).eval()
    return evaluate_stage2(model, val_loader, cfg, c2i, dev, run_dir=rd, ref_model_for_cka=ref_cka)


def recover_or_train(dataset, pipeline, arch, train_fn, cfg_yaml="tiny_imagenet.yaml",
                     baseline_ckpt=None, label=None):
    """Idempotent wrapper: skip if completed, re-eval if partial, else train."""
    tag = label or f"{dataset}/{pipeline}/{arch}"
    rd, m = find_completed_run(dataset, pipeline, arch)
    if rd and m:
        print(f"  ✓ {tag} recovered  top1={m['top1']:.4f}")
        return rd, m
    if rd and not m:
        print(f"  ⟳ {tag} best.pt found, re-evaluating …")
        m = ensure_eval(rd, arch, cfg_yaml, baseline_ckpt)
        print(f"  ✓ {tag} top1={m['top1']:.4f}")
        return rd, m
    print(f"  ▶ {tag} training …")
    rd, m = train_fn()
    print(f"  ✓ {tag} top1={m['top1']:.4f}")
    return rd, m


print("Orchestrator ready. Device:", orch.device)

Orchestrator ready. Device: cuda


## II. Migrations

Resolves `data/synthetic_sd` → `data/synthetic/tiny_imagenet` and links Stage 1 checkpoints under `results/`. **Does not regenerate** any synthetic images.

In [3]:
if RUN_MIGRATION:
    synth_path = orch.migrate_tiny_synthetic()
    print("Synthetic (Tiny) resolved to:", synth_path)
    orch.link_stage1_checkpoints()
    print("Legacy checkpoints linked under results/…/legacy/")
else:
    print("Skipping migrations.")

Synthetic (Tiny) resolved to: /mnt/data/cv/data/synthetic_sd
Legacy checkpoints linked under results/…/legacy/


## III. Validation sanity-check — Uniform 5× ResNet-18

One quick run to confirm the balanced-sampling fix works. If top-1 ≥ 40 %, the real signal is no longer buried under synthetic data. If < 40 %, **stop** — the synthetic images themselves may be bad.

In [4]:
if RUN_VALIDATION_CHECK:
    # Look up baseline top-1 to set a relative threshold
    _bl_rd, _bl_m = find_completed_run("tiny_imagenet", "baseline", "resnet18")
    if _bl_rd and _bl_m:
        _bl_top1 = _bl_m.get("top1", 0.40)
        _threshold = _bl_top1 - 0.05
        print(f"  Baseline top1={_bl_top1:.4f} → validation threshold={_threshold:.4f}")
    else:
        _threshold = 0.35
        print(f"  No baseline found — using fallback threshold={_threshold:.4f}")

    rd, m = recover_or_train(
        "tiny_imagenet", "uniform_5x", "resnet18",
        train_fn=lambda: orch.train_uniform("tiny_imagenet.yaml", "resnet18", 5),
        cfg_yaml="tiny_imagenet.yaml",
        label="VALIDATION: uniform_5x / resnet18",
    )
    val_acc = m.get("top1", 0.0)
    print(f"\n{'='*60}")
    if val_acc >= _threshold:
        print(f"  ✅ Validation PASSED — top1 = {val_acc:.4f} (≥ {_threshold:.4f})")
    else:
        print(f"  ❌ Validation FAILED — top1 = {val_acc:.4f} (< {_threshold:.4f})")
        print("     Balanced-sampling fix may not be working, or synthetic images are bad.")
        print("     Investigate before proceeding.")
    print(f"{'='*60}\n")
else:
    print("Skipping validation check.")

  No baseline found — using fallback threshold=0.3500
  ▶ VALIDATION: uniform_5x / resnet18 training …


/mnt/data/cv/src/training/stage2_train.py:43: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=mixed_precision)
Train:   0%|          | 0/469 [00:00<?, ?it/s]

/mnt/data/cv/src/training/train_eval.py:54: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1/30  train_loss=4.0061 acc=0.2749  val_loss=3.3490 acc=0.3553


Epoch 2/30  train_loss=2.9101 acc=0.4912  val_loss=2.9627 acc=0.4258


...

Epoch 27/30  train_loss=1.4692 acc=0.8650  val_loss=2.7580 acc=0.5135


Epoch 28/30  train_loss=1.4768 acc=0.8628  val_loss=2.7549 acc=0.5160


Epoch 29/30  train_loss=1.4688 acc=0.8664  val_loss=2.7475 acc=0.5150


/mnt/data/cv/src/stage2/orchestrator.py:186: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sd = torch.load(run_dir / "best.pt", map_location=self.device)


Epoch 30/30  train_loss=1.4608 acc=0.8675  val_loss=2.7505 acc=0.5146
  ✓ VALIDATION: uniform_5x / resnet18 top1=0.5165

  ✅ Validation PASSED — top1 = 0.5165 (≥ 0.3500)



## IV. Tiny ImageNet — ResNet-18 full grid (standard CE)

Order: **baseline** → uniform 5×/10×/15× → diagnostics extraction → utility targets → allocation CSVs → adaptive (hard_class / uncertainty / predicted_utility) → **ceiling**.

All runs use **standard cross-entropy**. Synthetic-aware loss variants can be added later (see § X).
Results go to `results/tiny_imagenet/{pipeline}/resnet18/{timestamp}/`.

In [5]:
if not RUN_TINY_R18:
    print("Skipping Tiny ImageNet ResNet-18 grid.")
else:
    ARCH = "resnet18"
    CFG = "tiny_imagenet.yaml"
    DS = "tiny_imagenet"
    tiny_r18_runs = {}  # pipeline -> (run_dir, metrics)

    # ── 1. Baseline ──────────────────────────────────────────────
    print("─── Baseline ───")
    rd, m = recover_or_train(
        DS, "baseline", ARCH,
        train_fn=lambda: orch.train_baseline(CFG, ARCH),
        cfg_yaml=CFG,
    )
    tiny_r18_runs["baseline"] = (rd, m)
    r18_ckpt = Path(rd) / "best.pt"

    # ── 2. Uniform 5× / 10× / 15× (standard CE) ────────────────
    # baseline_ckpt_same_arch=None → forces ConcatDataset + standard CE
    # regardless of synthetic_aware_loss in the YAML.
    print("\n─── Uniform scaling (standard CE) ───")
    for ratio in [5, 10, 15]:
        _r = ratio  # capture for lambda
        rd, m = recover_or_train(
            DS, f"uniform_{ratio}x", ARCH,
            train_fn=lambda _r=_r: orch.train_uniform(
                CFG, ARCH, _r, baseline_ckpt_same_arch=None,
            ),
            cfg_yaml=CFG, baseline_ckpt=r18_ckpt,
        )
        tiny_r18_runs[f"uniform_{ratio}x"] = (rd, m)

    # ── 3. Diagnostics + utility targets + allocations ───────────
    print("\n─── Diagnostics & allocations ───")
    cfg_t = orch.load_cfg(CFG)
    tr_t = get_train_transform(cfg_t.dataset.image_size)
    va_t = get_val_transform(cfg_t.dataset.image_size)
    _, _, c2i = get_baseline_loaders(cfg_t, tr_t, va_t)
    cids = class_ids_in_label_order(c2i)

    # utility: compare baseline vs uniform-15x per-class accuracy
    mb = tiny_r18_runs["baseline"][1]
    mu = tiny_r18_runs["uniform_15x"][1]
    util_t = orch.utility_from_metrics(mb, mu, cids)
    utility_path_t = PROJECT_ROOT / "results" / DS / "utility_from_uniform15x.json"
    utility_path_t.parent.mkdir(parents=True, exist_ok=True)
    utility_path_t.write_text(json.dumps(util_t, indent=2), encoding="utf-8")
    print("  Saved utility targets →", utility_path_t)

    # diagnostics CSV from baseline checkpoint
    diag_csv = PROJECT_ROOT / "results" / DS / "diagnostics" / ARCH / "class_diagnostics.csv"
    if diag_csv.is_file():
        print("  Diagnostics CSV recovered →", diag_csv)
    else:
        diag_csv = orch.compute_baseline_diagnostics(CFG, r18_ckpt, arch=ARCH, quality_csv=None)
        print("  Diagnostics CSV →", diag_csv)

    # allocation CSVs
    alloc_dir = PROJECT_ROOT / "results" / DS / "allocations"
    needed_policies = ["hard_class", "uncertainty", "predicted_utility"]
    existing = [p for p in needed_policies if (alloc_dir / f"allocation_{p}.csv").is_file()]
    if set(existing) == set(needed_policies):
        print("  Allocation CSVs recovered →", alloc_dir)
    else:
        orch.build_allocations(
            CFG, diag_csv,
            utility_path_t if utility_path_t.exists() else None,
            policies=needed_policies,
        )
        print("  Allocation CSVs built →", alloc_dir)

    # ── 4. Adaptive (standard CE) ────────────────────────────────
    # baseline_ckpt_same_arch=None → standard CE for all three policies.
    print("\n─── Adaptive (standard CE) ───")
    for pol in needed_policies:
        csvp = alloc_dir / f"allocation_{pol}.csv"
        if not csvp.is_file():
            print(f"  ⚠ Missing allocation CSV for {pol}, skipping")
            continue
        pipe_name = f"adaptive_15x_{pol}"
        rd, m = recover_or_train(
            DS, pipe_name, ARCH,
            train_fn=lambda _c=csvp, _n=pipe_name: orch.train_adaptive(
                CFG, ARCH, _c, name=_n, baseline_ckpt_same_arch=None,
            ),
            cfg_yaml=CFG, baseline_ckpt=r18_ckpt,
        )
        tiny_r18_runs[pipe_name] = (rd, m)

    # ── 5. Ceiling (100 % real) ──────────────────────────────────
    print("\n─── Ceiling ───")
    rd, m = recover_or_train(
        DS, "ceiling", ARCH,
        train_fn=lambda: orch.train_ceiling(CFG, ARCH, baseline_ckpt_same_arch=None),
        cfg_yaml=CFG, baseline_ckpt=r18_ckpt,
    )
    tiny_r18_runs["ceiling"] = (rd, m)

    # ── 6. Full evaluation pass on any runs missing metrics.json ─
    print("\n─── Evaluation sweep ───")
    root_t = PROJECT_ROOT / "results" / DS
    for pipe_dir in sorted(root_t.iterdir()):
        if not pipe_dir.is_dir() or pipe_dir.name in ("diagnostics", "allocations", "legacy", "fid_cache"):
            continue
        arch_dir = pipe_dir / ARCH
        if not arch_dir.is_dir():
            continue
        for run in sorted(arch_dir.iterdir()):
            if not run.is_dir() or not (run / "best.pt").is_file():
                continue
            if (run / "metrics.json").is_file():
                continue
            print(f"  Re-evaluating {pipe_dir.name}/{ARCH}/{run.name} …")
            ensure_eval(run, ARCH, CFG, baseline_ckpt=r18_ckpt)
            print(f"    ✓ metrics.json written")

    # ── aggregate index ──────────────────────────────────────────
    orch.aggregate_results_index(CFG)
    print("\n✅ Tiny ImageNet ResNet-18 grid complete. Results index updated.")

─── Baseline ───
  ▶ tiny_imagenet/baseline/resnet18 training …


/mnt/data/cv/src/training/stage2_train.py:43: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=mixed_precision)
Train:   0%|          | 0/79 [00:00<?, ?it/s]/mnt/data/cv/src/training/train_eval.py:54: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1/30  train_loss=5.2590 acc=0.0232  val_loss=4.7569 acc=0.0865


Epoch 2/30  train_loss=4.7354 acc=0.0894  val_loss=4.2026 acc=0.1963


Epoch 3/30  train_loss=4.3445 acc=0.1716  val_loss=3.8334 acc=0.2584


...

Epoch 27/30  train_loss=2.0713 acc=0.6847  val_loss=1.9545 acc=0.7089


Epoch 28/30  train_loss=2.0684 acc=0.6857  val_loss=1.9537 acc=0.7094


Epoch 29/30  train_loss=2.0559 acc=0.6876  val_loss=1.9515 acc=0.7093


/mnt/data/cv/src/stage2/orchestrator.py:186: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sd = torch.load(run_dir / "best.pt", map_location=self.device)


Epoch 30/30  train_loss=2.0608 acc=0.6876  val_loss=1.9501 acc=0.7099
  ✓ tiny_imagenet/ceiling/resnet18 top1=0.7099

─── Evaluation sweep ───

✅ Tiny ImageNet ResNet-18 grid complete. Results index updated.


## V. Tiny ImageNet — MobileNetV3-Small

Subset of the grid: **baseline** → **uniform 15×** → **adaptive predicted_utility 15×** → **ceiling**.

In [ ]:
if not RUN_TINY_MOBILENET:
    print("Skipping Tiny ImageNet MobileNetV3-Small.")
else:
    ARCH_M = "mobilenet_v3_small"
    CFG = "tiny_imagenet.yaml"
    DS = "tiny_imagenet"
    mob_runs = {}

    # ── Baseline ─────────────────────────────────────────────────
    print("─── Baseline ───")
    rd, m = recover_or_train(
        DS, "baseline", ARCH_M,
        train_fn=lambda: orch.train_baseline(CFG, ARCH_M),
        cfg_yaml=CFG,
    )
    mob_runs["baseline"] = (rd, m)
    mob_ckpt = Path(rd) / "best.pt"

    # ── Uniform 15× (standard CE) ────────────────────────────────
    print("\n─── Uniform 15× (standard CE) ───")
    rd, m = recover_or_train(
        DS, "uniform_15x", ARCH_M,
        train_fn=lambda: orch.train_uniform(
            CFG, ARCH_M, 15, baseline_ckpt_same_arch=None,
        ),
        cfg_yaml=CFG, baseline_ckpt=mob_ckpt,
    )
    mob_runs["uniform_15x"] = (rd, m)

    # ── Adaptive predicted_utility 15× (standard CE) ─────────────
    print("\n─── Adaptive predicted_utility (standard CE) ───")
    alloc_csv = PROJECT_ROOT / "results" / DS / "allocations" / "allocation_predicted_utility.csv"
    if alloc_csv.is_file():
        rd, m = recover_or_train(
            DS, "adaptive_15x_predicted_utility", ARCH_M,
            train_fn=lambda: orch.train_adaptive(
                CFG, ARCH_M, alloc_csv,
                name="adaptive_15x_predicted_utility",
                baseline_ckpt_same_arch=None,
            ),
            cfg_yaml=CFG, baseline_ckpt=mob_ckpt,
        )
        mob_runs["adaptive_15x_predicted_utility"] = (rd, m)
    else:
        print("  ⚠ Allocation CSV missing — run Tiny R18 grid first")

    # ── Ceiling ──────────────────────────────────────────────────
    print("\n─── Ceiling ───")
    rd, m = recover_or_train(
        DS, "ceiling", ARCH_M,
        train_fn=lambda: orch.train_ceiling(CFG, ARCH_M, baseline_ckpt_same_arch=None),
        cfg_yaml=CFG, baseline_ckpt=mob_ckpt,
    )
    mob_runs["ceiling"] = (rd, m)

    # ── Evaluation sweep ─────────────────────────────────────────
    print("\n─── Evaluation sweep ───")
    root_t = PROJECT_ROOT / "results" / DS
    for pipe_dir in sorted(root_t.iterdir()):
        if not pipe_dir.is_dir() or pipe_dir.name in ("diagnostics", "allocations", "legacy", "fid_cache"):
            continue
        arch_dir = pipe_dir / ARCH_M
        if not arch_dir.is_dir():
            continue
        for run in sorted(arch_dir.iterdir()):
            if not run.is_dir() or not (run / "best.pt").is_file():
                continue
            if (run / "metrics.json").is_file():
                continue
            print(f"  Re-evaluating {pipe_dir.name}/{ARCH_M}/{run.name} …")
            ensure_eval(run, ARCH_M, CFG, baseline_ckpt=mob_ckpt)
            print(f"    ✓ metrics.json written")

    orch.aggregate_results_index(CFG)
    print("\n✅ Tiny ImageNet MobileNetV3-Small complete. Results index updated.")

/mnt/data/cv/src/training/stage2_train.py:43: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=mixed_precision)


─── Baseline ───
  ▶ tiny_imagenet/baseline/mobilenet_v3_small training …


Train:   0%|          | 0/79 [00:00<?, ?it/s]/mnt/data/cv/src/training/train_eval.py:54: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1/30  train_loss=5.2901 acc=0.0104  val_loss=5.1561 acc=0.0278


Epoch 2/30  train_loss=5.0903 acc=0.0410  val_loss=4.7978 acc=0.0997


...

Epoch 6/30  train_loss=2.6327 acc=0.5206  val_loss=2.0333 acc=0.6801


Epoch 7/30  train_loss=2.6026 acc=0.5277  val_loss=2.0084 acc=0.6838


Epoch 8/30  train_loss=2.5669 acc=0.5375  val_loss=1.9994 acc=0.6881


Epoch 9/30  train_loss=2.5344 acc=0.5454  val_loss=1.9811 acc=0.6947


Train:  41%|████      | 637/1563 [05:17<07:52,  1.96it/s]

## VI. CIFAR-100 — ResNet-18

**Baseline** → **uniform 15×** → diagnostics + utility + allocations → **adaptive predicted_utility 15×** → **ceiling**.

Uses the policy from `configs/cifar100.yaml → scope.cifar_adaptive_policy` (default: `predicted_utility`).

In [ ]:
if not RUN_CIFAR:
    print("Skipping CIFAR-100 experiments.")
else:
    ARCH_C = "resnet18"
    CFG_C = "cifar100.yaml"
    DS_C = "cifar100"
    cifar_runs = {}

    # ── Baseline ─────────────────────────────────────────────────
    print("─── Baseline ───")
    rd, m = recover_or_train(
        DS_C, "baseline", ARCH_C,
        train_fn=lambda: orch.train_baseline(CFG_C, ARCH_C),
        cfg_yaml=CFG_C,
    )
    cifar_runs["baseline"] = (rd, m)
    c_ckpt = Path(rd) / "best.pt"

    # ── Uniform 15× (standard CE) ────────────────────────────────
    print("\n─── Uniform 15× (standard CE) ───")
    rd, m = recover_or_train(
        DS_C, "uniform_15x", ARCH_C,
        train_fn=lambda: orch.train_uniform(
            CFG_C, ARCH_C, 15, baseline_ckpt_same_arch=None,
        ),
        cfg_yaml=CFG_C, baseline_ckpt=c_ckpt,
    )
    cifar_runs["uniform_15x"] = (rd, m)

    # ── Diagnostics + utility + allocations ──────────────────────
    print("\n─── Diagnostics & allocations ───")
    cfg_c = orch.load_cfg(CFG_C)
    tr_t = get_train_transform(cfg_c.dataset.image_size)
    va_t = get_val_transform(cfg_c.dataset.image_size)
    _, _, c2i_c = get_baseline_loaders(cfg_c, tr_t, va_t)
    cids_c = class_ids_in_label_order(c2i_c)

    mb_c = cifar_runs["baseline"][1]
    mu_c = cifar_runs["uniform_15x"][1]
    util_c = orch.utility_from_metrics(mb_c, mu_c, cids_c)
    utility_path_c = PROJECT_ROOT / "results" / DS_C / "utility_from_uniform15x.json"
    utility_path_c.parent.mkdir(parents=True, exist_ok=True)
    utility_path_c.write_text(json.dumps(util_c, indent=2), encoding="utf-8")
    print("  Saved CIFAR utility targets →", utility_path_c)

    diag_csv_c = PROJECT_ROOT / "results" / DS_C / "diagnostics" / ARCH_C / "class_diagnostics.csv"
    if diag_csv_c.is_file():
        print("  Diagnostics CSV recovered →", diag_csv_c)
    else:
        diag_csv_c = orch.compute_baseline_diagnostics(CFG_C, c_ckpt, arch=ARCH_C)
        print("  Diagnostics CSV →", diag_csv_c)

    alloc_policies_c = ["hard_class", "predicted_utility"]
    alloc_dir_c = PROJECT_ROOT / "results" / DS_C / "allocations"
    existing_c = [p for p in alloc_policies_c if (alloc_dir_c / f"allocation_{p}.csv").is_file()]
    if set(existing_c) == set(alloc_policies_c):
        print("  Allocation CSVs recovered →", alloc_dir_c)
    else:
        orch.build_allocations(
            CFG_C, diag_csv_c,
            utility_path_c if utility_path_c.exists() else None,
            policies=alloc_policies_c,
        )
        print("  Allocation CSVs built →", alloc_dir_c)

    # ── Adaptive (standard CE) ───────────────────────────────────
    print("\n─── Adaptive (standard CE) ───")
    pol_c = cfg_c.scope.cifar_adaptive_policy  # default: predicted_utility
    acsv_c = alloc_dir_c / f"allocation_{pol_c}.csv"
    pipe_c = f"adaptive_15x_{pol_c}"
    if acsv_c.is_file():
        rd, m = recover_or_train(
            DS_C, pipe_c, ARCH_C,
            train_fn=lambda: orch.train_adaptive(
                CFG_C, ARCH_C, acsv_c, name=pipe_c, baseline_ckpt_same_arch=None,
            ),
            cfg_yaml=CFG_C, baseline_ckpt=c_ckpt,
        )
        cifar_runs[pipe_c] = (rd, m)
    else:
        print(f"  ⚠ Missing allocation CSV: {acsv_c}")

    # ── Ceiling ──────────────────────────────────────────────────
    print("\n─── Ceiling ───")
    rd, m = recover_or_train(
        DS_C, "ceiling", ARCH_C,
        train_fn=lambda: orch.train_ceiling(CFG_C, ARCH_C, baseline_ckpt_same_arch=None),
        cfg_yaml=CFG_C, baseline_ckpt=c_ckpt,
    )
    cifar_runs["ceiling"] = (rd, m)

    orch.aggregate_results_index(CFG_C)
    print("\n✅ CIFAR-100 ResNet-18 track complete. Results index updated.")

## VII. FID — Global (cached)

Reads `results/{dataset}/fid_cache/fid_summary.json` if it exists. Only recomputes if the cache is missing. **Never regenerates** synthetic images.

In [ ]:
if not RUN_FID:
    print("Skipping FID.")
else:
    for ds_name, yaml_name, ratios in [
        ("tiny_imagenet", "tiny_imagenet.yaml", [5, 10, 15]),
        ("cifar100", "cifar100.yaml", [15]),
    ]:
        cache_json = PROJECT_ROOT / "results" / ds_name / "fid_cache" / "fid_summary.json"
        if cache_json.is_file():
            fid_data = json.loads(cache_json.read_text(encoding="utf-8"))
            print(f"  {ds_name} FID (cached): {fid_data}")
        else:
            print(f"  {ds_name} FID cache not found — computing …")
            fid_data = orch.compute_global_fid(yaml_name, ratios=ratios)
            print(f"  {ds_name} FID: {fid_data}")

## VIII. Figures

Reads `results/*/results_index.json` and writes comparison plots to `figures/stage2/` (PNG + PDF).

In [ ]:
if not RUN_FIGURES:
    print("Skipping figures.")
else:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig_dir = PROJECT_ROOT / "figures" / "stage2"
    fig_dir.mkdir(parents=True, exist_ok=True)

    for ds_name, yaml_name in [("tiny_imagenet", "tiny_imagenet.yaml"), ("cifar100", "cifar100.yaml")]:
        idx_path = PROJECT_ROOT / "results" / ds_name / "results_index.json"
        if not idx_path.is_file():
            print(f"  No results index for {ds_name} — skipping.")
            continue

        rows = json.loads(idx_path.read_text(encoding="utf-8"))
        if not rows:
            print(f"  Empty results index for {ds_name} — skipping.")
            continue

        # Keep only the last (latest) run per (pipeline, arch)
        best = {}
        for r in rows:
            key = (r["pipeline"], r["arch"])
            best[key] = r

        keys_sorted = sorted(best.keys())
        labels = [f"{k[0]}\n{k[1]}" for k in keys_sorted]
        vals = [best[k]["top1"] * 100 for k in keys_sorted]  # percent

        # ── Bar chart: top-1 per pipeline / arch ────────────────
        fig, ax = plt.subplots(figsize=(max(8, len(labels) * 0.9), 5))
        bars = ax.bar(range(len(labels)), vals, color="steelblue", edgecolor="white", linewidth=0.5)
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels, rotation=55, ha="right", fontsize=8)
        ax.set_ylabel("Val Top-1 (%)")
        ax.set_title(f"{ds_name} — Stage 2 Top-1 Accuracy")
        ax.set_ylim(0, max(vals) * 1.15 if vals else 100)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                    f"{v:.1f}", ha="center", va="bottom", fontsize=7)
        plt.tight_layout()
        out_png = fig_dir / f"{ds_name}_summary_top1.png"
        plt.savefig(out_png, dpi=300)
        plt.savefig(out_png.with_suffix(".pdf"))
        plt.close()
        print(f"  Saved {out_png.name} + .pdf")

    # ── FID vs budget plot (Tiny only) ───────────────────────────
    fid_json = PROJECT_ROOT / "results" / "tiny_imagenet" / "fid_cache" / "fid_summary.json"
    if fid_json.is_file():
        fid = json.loads(fid_json.read_text(encoding="utf-8"))
        budgets, scores = [], []
        for r in [5, 10, 15]:
            key = f"fid_{r}x"
            if key in fid and fid[key] is not None:
                budgets.append(r)
                scores.append(fid[key])
        if budgets:
            fig, ax = plt.subplots(figsize=(5, 3.5))
            ax.plot(budgets, scores, "o-", color="coral", linewidth=2)
            ax.set_xlabel("Synthetic budget (×)")
            ax.set_ylabel("FID (lower = better)")
            ax.set_title("Tiny ImageNet — Global FID vs budget")
            ax.set_xticks(budgets)
            plt.tight_layout()
            out_fid = fig_dir / "tiny_fid_vs_budget.png"
            plt.savefig(out_fid, dpi=300)
            plt.savefig(out_fid.with_suffix(".pdf"))
            plt.close()
            print(f"  Saved {out_fid.name} + .pdf")

    # ── Summary table to console ─────────────────────────────────
    print("\n" + "=" * 72)
    print("SUMMARY TABLE")
    print("=" * 72)
    for ds_name in ["tiny_imagenet", "cifar100"]:
        idx_path = PROJECT_ROOT / "results" / ds_name / "results_index.json"
        if not idx_path.is_file():
            continue
        rows = json.loads(idx_path.read_text(encoding="utf-8"))
        best = {}
        for r in rows:
            key = (r["pipeline"], r["arch"])
            best[key] = r
        print(f"\n  {ds_name}")
        print(f"  {'Pipeline':<40} {'Arch':<22} {'Top-1':>8}")
        print(f"  {'-'*40} {'-'*22} {'-'*8}")
        for k in sorted(best.keys()):
            r = best[k]
            print(f"  {r['pipeline']:<40} {r['arch']:<22} {r['top1']*100:>7.2f}%")
    print("=" * 72)
    print("\n✅ All figures saved to", fig_dir)

## IX. Synthetic-aware loss variants (optional — `RUN_SA_VARIANTS`)

Set `RUN_SA_VARIANTS = True` in cell 2, then **Run All** again. Every standard-CE run above is already complete so it will be skipped instantly; only the `_sa` pipelines below will train.

These reuse the **same allocation CSVs** but pass the baseline checkpoint to enable `RealSyntheticMixDataset` + distance-weighted CE (`synthetic_aware_loss: true` in YAML).

In [ ]:
if not RUN_SA_VARIANTS:
    print("Skipping synthetic-aware loss variants (RUN_SA_VARIANTS = False).")
else:
    # ── Tiny ImageNet R18: adaptive predicted_utility + SA loss ──
    print("─── Tiny: adaptive predicted_utility + SA loss ───")
    _cfg_t = "tiny_imagenet.yaml"
    _ds_t = "tiny_imagenet"
    _arch_t = "resnet18"

    # Need baseline checkpoint for SA loss centroids + CKA
    _bl_rd, _ = find_completed_run(_ds_t, "baseline", _arch_t)
    _r18_ckpt = Path(_bl_rd) / "best.pt" if _bl_rd else None

    _sa_csv_t = PROJECT_ROOT / "results" / _ds_t / "allocations" / "allocation_predicted_utility.csv"
    _sa_pipe_t = "adaptive_15x_predicted_utility_sa"

    if _r18_ckpt and _r18_ckpt.is_file() and _sa_csv_t.is_file():
        rd, m = recover_or_train(
            _ds_t, _sa_pipe_t, _arch_t,
            train_fn=lambda: orch.train_adaptive(
                _cfg_t, _arch_t, _sa_csv_t,
                name=_sa_pipe_t, baseline_ckpt_same_arch=_r18_ckpt,
            ),
            cfg_yaml=_cfg_t, baseline_ckpt=_r18_ckpt,
        )
    else:
        print("  ⚠ Missing baseline checkpoint or allocation CSV — run standard grid first")

    # ── Tiny ImageNet R18: uniform 15× + SA loss ────────────────
    print("\n─── Tiny: uniform 15× + SA loss ───")
    _u15_sa = "uniform_15x_sa"
    if _r18_ckpt and _r18_ckpt.is_file():
        rd, m = recover_or_train(
            _ds_t, _u15_sa, _arch_t,
            train_fn=lambda: orch.train_uniform(
                _cfg_t, _arch_t, 15,
                baseline_ckpt_same_arch=_r18_ckpt,
                name=_u15_sa,
            ),
            cfg_yaml=_cfg_t, baseline_ckpt=_r18_ckpt,
        )
    else:
        print("  ⚠ Missing baseline checkpoint — run standard grid first")

    # ── CIFAR-100 R18: adaptive + SA loss ────────────────────────
    print("\n─── CIFAR: adaptive predicted_utility + SA loss ───")
    _cfg_c = "cifar100.yaml"
    _ds_c = "cifar100"
    _arch_c = "resnet18"

    _bl_rd_c, _ = find_completed_run(_ds_c, "baseline", _arch_c)
    _c_ckpt = Path(_bl_rd_c) / "best.pt" if _bl_rd_c else None

    _cfg_obj_c = orch.load_cfg(_cfg_c)
    _pol_c = _cfg_obj_c.scope.cifar_adaptive_policy
    _sa_csv_c = PROJECT_ROOT / "results" / _ds_c / "allocations" / f"allocation_{_pol_c}.csv"
    _sa_pipe_c = f"adaptive_15x_{_pol_c}_sa"

    if _c_ckpt and _c_ckpt.is_file() and _sa_csv_c.is_file():
        rd, m = recover_or_train(
            _ds_c, _sa_pipe_c, _arch_c,
            train_fn=lambda: orch.train_adaptive(
                _cfg_c, _arch_c, _sa_csv_c,
                name=_sa_pipe_c, baseline_ckpt_same_arch=_c_ckpt,
            ),
            cfg_yaml=_cfg_c, baseline_ckpt=_c_ckpt,
        )
    else:
        print("  ⚠ Missing baseline checkpoint or allocation CSV — run CIFAR grid first")

    # Re-aggregate both indexes to include _sa runs
    orch.aggregate_results_index("tiny_imagenet.yaml")
    orch.aggregate_results_index("cifar100.yaml")
    print("\n✅ SA variants complete. Results indexes updated.")

## X. Submission checklist

- [ ] `results/` contains per-run `metrics.json`, `best.pt`, `training_curves.json`
- [ ] Figures under `figures/stage2/` (PNG + PDF)
- [ ] Summary table above matches expected numbers
- [ ] FID scores recorded in `results/*/fid_cache/fid_summary.json`
- [ ] *(optional)* SA variants trained (`RUN_SA_VARIANTS = True`)
- [ ] Export Stage 2 report PDF with numbers filled from `metrics.json` / `results_index.json`